In [1]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
!pip install -q transformers torch pandas scikit-learn


In [2]:

# ============================================================
# CELL 2: Imports
# ============================================================
import os
import re
import json
import glob
import shutil
import zipfile
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import pandas as pd
from transformers import AutoTokenizer, AutoModel, AutoConfig
from transformers.modeling_outputs import ModelOutput
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    mean_squared_error,
    mean_absolute_error,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


Using device: cuda


In [3]:

# ============================================================
# CELL 3: Load the trained model checkpoint from Google Drive
#
# files.upload() (the browser upload widget) is unreliable for large
# files like a ~400MB model checkpoint — it can stall for a long time
# with no progress feedback. Uploading to Drive first (drag-and-drop
# in your browser, which is fast and resumable) and mounting Drive
# here instead is far more reliable.
#
# Before running this cell:
#   1. Upload your trained model zip (the zipped "best/" directory —
#      containing config.json, pytorch_model.bin, tokenizer files,
#      and training_meta.json) to your Google Drive, e.g. into
#      "My Drive/colab_models/".
#   2. Set MODEL_ZIP_DRIVE_PATH below to that file's path.
# ============================================================
from google.colab import drive

drive.mount("/content/drive")

MODEL_ZIP_DRIVE_PATH = "/content/drive/MyDrive/roberta-base_model.zip"


assert os.path.exists(MODEL_ZIP_DRIVE_PATH), (
    f"Model zip not found at {MODEL_ZIP_DRIVE_PATH} — check the path matches "
    "where you uploaded it in your Google Drive."
)

MODEL_ZIP_PATH = "/content/eval_model.zip"
shutil.copy(MODEL_ZIP_DRIVE_PATH, MODEL_ZIP_PATH)
print(f"Copied model checkpoint from Drive ({os.path.getsize(MODEL_ZIP_PATH) / 1e6:.1f} MB).")

MODEL_DIR = "/content/eval_model"
if os.path.exists(MODEL_DIR):
    shutil.rmtree(MODEL_DIR)
os.makedirs(MODEL_DIR, exist_ok=True)

with zipfile.ZipFile(MODEL_ZIP_PATH, "r") as zf:
    zf.extractall(MODEL_DIR)

# The zip may extract straight into MODEL_DIR, or into a single
# subfolder (e.g. "best/") depending on how it was zipped — detect
# and flatten either layout so config.json is always at MODEL_DIR root.
if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    subdirs = [
        d for d in glob.glob(os.path.join(MODEL_DIR, "*"))
        if os.path.isdir(d)
    ]
    if len(subdirs) == 1 and os.path.exists(os.path.join(subdirs[0], "config.json")):
        for item in os.listdir(subdirs[0]):
            shutil.move(os.path.join(subdirs[0], item), MODEL_DIR)
        shutil.rmtree(subdirs[0])

assert os.path.exists(os.path.join(MODEL_DIR, "config.json")), (
    f"config.json not found in {MODEL_DIR} — check the uploaded zip contains "
    "the checkpoint files directly (config.json, pytorch_model.bin, tokenizer files)."
)
print(f"Model checkpoint ready at: {MODEL_DIR}")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied model checkpoint from Drive (416.2 MB).
Model checkpoint ready at: /content/eval_model


In [4]:

# ============================================================
# CELL 4: Upload dataset CSV(s) and lexicon.json
#
# Upload one or more dataset_match_*.csv files (multi-select is fine)
# AND commentary_analysis_system/data/lexicon.json. These are small
# text files (a few MB at most), so the files.upload() widget is fine
# here — it's only large binaries like the model checkpoint that need
# the Drive-mount approach used in Cell 3.
# ============================================================
from google.colab import files

print("Upload one or more dataset_match_*.csv files, plus lexicon.json:")
uploaded_data = files.upload()

DATASET_CSV_PATHS = sorted(
    name for name in uploaded_data.keys() if name.lower().endswith(".csv")
)
LEXICON_JSON_PATH = next(
    (name for name in uploaded_data.keys() if name.lower().endswith("lexicon.json")),
    None,
)

assert DATASET_CSV_PATHS, "No CSV files were uploaded."
assert LEXICON_JSON_PATH, "lexicon.json was not uploaded."
print(f"Datasets ({len(DATASET_CSV_PATHS)}): {DATASET_CSV_PATHS}")
print(f"Lexicon config: {LEXICON_JSON_PATH}")



Upload one or more dataset_match_*.csv files, plus lexicon.json:


Saving dataset_match_1_Geo_vs_POR_WCA_2023.csv to dataset_match_1_Geo_vs_POR_WCA_2023.csv
Saving lexicon.json to lexicon.json
Datasets (1): ['dataset_match_1_Geo_vs_POR_WCA_2023.csv']
Lexicon config: lexicon.json


In [5]:

# ============================================================
# CELL 5: DualHeadRoBERTa model definition
#
# Plain nn.Linear heads — matches colab_backend.py / colab
# testing/colab_training.py (the scripts that actually produced every
# trained checkpoint in this project), not the Sequential(Dropout,
# Linear) variant used by the local pipeline's transformer_classifier.py.
# ============================================================
@dataclass
class DualHeadRoBERTaOutput(ModelOutput):
    loss: Optional[torch.FloatTensor] = None
    logits: torch.FloatTensor = None
    highlight_scores: torch.FloatTensor = None


class DualHeadRoBERTa(nn.Module):
    def __init__(self, model_name, num_labels, id2label, label2id):
        super().__init__()
        self.config = AutoModel.from_pretrained(model_name).config
        self.config.num_labels = num_labels
        self.config.id2label = id2label
        self.config.label2id = label2id

        self.roberta = AutoModel.from_pretrained(model_name)

        self.event_classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.highlight_scorer = nn.Linear(self.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = getattr(outputs, "pooler_output", None)
        if pooled_output is None:
            pooled_output = outputs.last_hidden_state[:, 0, :]

        event_logits = self.event_classifier(pooled_output)
        highlight_score = torch.sigmoid(self.highlight_scorer(pooled_output))

        return DualHeadRoBERTaOutput(logits=event_logits, highlight_scores=highlight_score)



In [6]:

# ============================================================
# CELL 6: Load the checkpoint
#
# Reads id2label/label2id/num_labels from config.json, and
# window_size/tokenizer_max_length from training_meta.json (falling
# back to config attributes, then to window_size=1 /
# tokenizer_max_length=128 if neither is present).
# ============================================================
class LoadedModel:
    def __init__(self, model_dir):
        config = AutoConfig.from_pretrained(model_dir)
        self.num_labels = config.num_labels
        self.id2label = {int(k): v for k, v in config.id2label.items()}
        self.label2id = {k: int(v) for k, v in config.label2id.items()}

        self.window_size = getattr(config, "window_size", 1)
        self.tokenizer_max_length = getattr(config, "tokenizer_max_length", 128)

        meta_path = os.path.join(model_dir, "training_meta.json")
        if os.path.exists(meta_path):
            with open(meta_path, "r") as f:
                meta = json.load(f)
            self.window_size = meta.get("window_size", self.window_size)
            self.tokenizer_max_length = meta.get("tokenizer_max_length", self.tokenizer_max_length)

        base_model_name = getattr(config, "_name_or_path", "roberta-base")
        self.model = DualHeadRoBERTa(base_model_name, self.num_labels, self.id2label, self.label2id)

        state_dict = torch.load(os.path.join(model_dir, "pytorch_model.bin"), map_location="cpu")
        self.model.load_state_dict(state_dict)
        self.model.to(DEVICE)
        self.model.eval()

        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)

    def predict_probs(self, text_list, batch_size=32):
        """Returns (event_probs, highlight_scores) — one list of 10-way
        softmax probs and one highlight-score float per input text."""
        all_event_probs, all_highlight_scores = [], []
        for i in range(0, len(text_list), batch_size):
            batch_texts = text_list[i:i + batch_size]
            inputs = self.tokenizer(
                batch_texts, padding=True, truncation=True,
                max_length=self.tokenizer_max_length, return_tensors="pt",
            )
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = self.model(**inputs)
                probs = torch.softmax(outputs.logits, dim=-1)
                all_event_probs.extend(probs.tolist())
                all_highlight_scores.extend(outputs.highlight_scores.view(-1).tolist())
        return all_event_probs, all_highlight_scores


loaded_model = LoadedModel(MODEL_DIR)
print(f"Loaded model: num_labels={loaded_model.num_labels}, "
      f"window_size={loaded_model.window_size}, "
      f"tokenizer_max_length={loaded_model.tokenizer_max_length}")
print(f"id2label: {loaded_model.id2label}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: /content/eval_model
Key                     | Status     |  | 
------------------------+------------+--+-
highlight_scorer.bias   | UNEXPECTED |  | 
event_classifier.weight | UNEXPECTED |  | 
event_classifier.bias   | UNEXPECTED |  | 
highlight_scorer.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: /content/eval_model
Key                     | Status     |  | 
------------------------+------------+--+-
highlight_scorer.bias   | UNEXPECTED |  | 
event_classifier.weight | UNEXPECTED |  | 
event_classifier.bias   | UNEXPECTED |  | 
highlight_scorer.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: num_labels=10, window_size=5, tokenizer_max_length=256
id2label: {0: 'normal_play', 1: 'try', 2: 'goal_kick', 3: 'card_event', 4: 'scrum', 5: 'maul', 6: 'lineout', 7: 'kick_off', 8: 'tmo_replay', 9: 'penalty'}


In [7]:

# ============================================================
# CELL 7: Sliding-window context builder
#
# Ported from commentary_analysis_system/src/data_processing/chunker.py.
# The ML/Hybrid model was trained on windowed text (each row joined
# with its neighbors) — skipping this for a checkpoint with
# window_size > 1 collapses predictions to a single class.
# ============================================================
def build_context_windows(texts: list[str], window_size: int) -> list[str]:
    if window_size <= 1:
        return list(texts)
    half = window_size // 2
    n = len(texts)
    return [
        " ".join(texts[max(0, i - half): min(n, i + half + 1)])
        for i in range(n)
    ]



In [8]:

# ============================================================
# CELL 8: Lexicon model
#
# Ported verbatim from
# commentary_analysis_system/src/models/lexicon_model.py.
# ============================================================
class LexiconModel:
    def __init__(self, config_path):
        with open(config_path, "r") as f:
            self.config = json.load(f)

    def _count_matches_weighted(self, text: str, terms: list[dict]) -> float:
        score = 0.0
        negations = [
            "no", "not", "missed", "missed the",
            "almost", "nearly", "just", "prevented",
            "stopped", "denied", "failed", "attempt",
            "trying to", "going for", "couldn't", "could not",
            "short of", "held up", "short",
        ]
        for term_obj in terms:
            if isinstance(term_obj, str):
                term, weight = term_obj, 1.0
            else:
                term = term_obj.get("text", "")
                weight = float(term_obj.get("weight", 1.0))
            if not term:
                continue

            if "\\" in term:
                pattern = term
            else:
                escaped_term = re.escape(term)
                pattern = rf"\b{escaped_term}(?:s|ed|ing|d)?\b"

            for match in re.finditer(pattern, text, re.IGNORECASE):
                start_idx = match.start()
                preceding_text = text[max(0, start_idx - 15):start_idx].lower()
                is_negated = any(preceding_text.strip().endswith(neg) for neg in negations)
                if not is_negated:
                    score += weight
        return score

    def generate_features(self, text: str) -> dict:
        text = text.lower()
        features = {}
        for category in self.config.get("categories", []):
            cat_id = category["id"]
            features[cat_id] = self._count_matches_weighted(text, category.get("terms", []))
        return features

    def score_chunk(self, text: str) -> float:
        feats = self.generate_features(text)
        raw_score = 0.0
        for category in self.config.get("categories", []):
            cat_id = category["id"]
            weight = category.get("weight", 0.0)
            count = feats.get(cat_id, 0)
            scaled_count = float(count) * 100.0
            raw_score += min(1.0, scaled_count) * weight
        return min(raw_score, 1.0)


lexicon_model = LexiconModel(LEXICON_JSON_PATH)
print(f"Loaded lexicon with {len(lexicon_model.config.get('categories', []))} categories.")


Loaded lexicon with 10 categories.


In [9]:

# ============================================================
# CELL 9: Hybrid combination
#
# Ported verbatim (formula-for-formula) from
# commentary_analysis_system/src/models/hybrid_model.py's
# HybridModel.predict(). Default weights match this project's
# production config (data/config/settings.json).
# ============================================================
HIGHLIGHT_LEXICON_WEIGHT = 0.75
EVENT_LEXICON_WEIGHT = 0.35
ROBERTA_WEIGHT = 1.0


def run_all_approaches(texts: list[str], id2label: dict, model: LoadedModel, lexicon: LexiconModel):
    """
    texts: raw per-row commentary text, in original CSV row order.
    Returns a dict of per-row predictions for all 3 approaches.
    """
    windowed_texts = build_context_windows(texts, model.window_size)
    roberta_event_probs, roberta_highlight_scores = model.predict_probs(windowed_texts)

    ml_pred_event_id, ml_highlight_score = [], []
    lex_pred_event_id, lex_score = [], []
    hybrid_pred_event_id, hybrid_score_list = [], []

    for text, r_probs, r_h_score in zip(texts, roberta_event_probs, roberta_highlight_scores):
        # --- ML Model Only ---
        ml_event_id = max(range(len(r_probs)), key=lambda i: r_probs[i])
        ml_pred_event_id.append(ml_event_id)
        ml_highlight_score.append(r_h_score)

        # --- Lexicon Only ---
        l_prob = lexicon.score_chunk(text)
        lexicon_features = lexicon.generate_features(text)
        max_lex_score, lex_event_id = -1.0, 0
        for i in range(len(r_probs)):
            event_name = id2label.get(i, "normal_play")
            l_score = lexicon_features.get(event_name, 0.0)
            if l_score > max_lex_score:
                max_lex_score, lex_event_id = l_score, i
        lex_pred_event_id.append(lex_event_id)
        lex_score.append(l_prob)

        # --- Hybrid ---
        hybrid_score = min(1.0, (r_h_score * ROBERTA_WEIGHT) + (l_prob * HIGHLIGHT_LEXICON_WEIGHT))
        hybrid_score_list.append(hybrid_score)

        max_hybrid_prob, hyb_event_id = -1.0, 0
        for i, p in enumerate(r_probs):
            event_name = id2label.get(i, "normal_play")
            l_prob_for_event = lexicon_features.get(event_name, 0.0)
            l_prob_normalized = min(1.0, float(l_prob_for_event) * 100.0)
            event_hybrid_prob = (p * ROBERTA_WEIGHT) + (l_prob_normalized * EVENT_LEXICON_WEIGHT)
            if event_hybrid_prob > max_hybrid_prob:
                max_hybrid_prob, hyb_event_id = event_hybrid_prob, i
        hybrid_pred_event_id.append(hyb_event_id)

    return {
        "ML Model Only": {"event_id": ml_pred_event_id, "score": ml_highlight_score},
        "Lexicon Only": {"event_id": lex_pred_event_id, "score": lex_score},
        "Hybrid Model": {"event_id": hybrid_pred_event_id, "score": hybrid_score_list},
    }



In [10]:

# ============================================================
# CELL 10: Ground-truth extraction from a dataset CSV
#
# Handles both current ("event_class"/"highlight_score") and older
# ("event"/"score") column names, same fallback as
# build_windowed_dataframe() in colab_backend.py and _extract_labels()
# in the local pipeline's evaluator.py: missing/blank event -> normal_play.
# ============================================================
def load_dataset_csv(csv_path: str, label2id: dict):
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=["text"]).reset_index(drop=True)

    event_col = "event_class" if "event_class" in df.columns else "event"
    score_col = "highlight_score" if "highlight_score" in df.columns else "score"

    df["true_event_id"] = df[event_col].apply(
        lambda x: label2id.get(str(x).strip(), 0) if pd.notna(x) and str(x).strip() else 0
    )
    df["true_score"] = pd.to_numeric(df[score_col], errors="coerce").fillna(0.0)

    return df


In [11]:

# ============================================================
# CELL 11: Metrics computation
#
# Mirrors make_compute_metrics() in colab_backend.py / colab
# testing/colab_training.py — sklearn weighted + macro averages over
# all classes (including normal_play), plus MSE/MAE for regression.
# ============================================================
def compute_classification_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
    }


def compute_regression_metrics(y_true, y_pred):
    return {
        "mse": mean_squared_error(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
    }


def print_metrics_table(title, metrics_by_approach, keys):
    print(f"\n--- {title} ---")
    header = f"{'Approach':<16}" + "".join(f"{k:>16}" for k in keys)
    print(header)
    print("-" * len(header))
    for approach, metrics in metrics_by_approach.items():
        row = f"{approach:<16}" + "".join(f"{metrics[k]:>16.4f}" for k in keys)
        print(row)



In [12]:

# ============================================================
# CELL 12: Run evaluation over all uploaded datasets
#
# Computes per-match metrics for all 3 approaches, then pools
# predictions across every match for a combined/aggregate result.
# ============================================================
CLASSIFICATION_KEYS = ["accuracy", "precision", "recall", "f1", "precision_macro", "recall_macro", "f1_macro"]
REGRESSION_KEYS = ["mse", "mae"]
APPROACH_NAMES = ["ML Model Only", "Lexicon Only", "Hybrid Model"]

per_match_results = {}  # match_name -> {"classification": {...}, "regression": {...}}
pooled_true_event, pooled_true_score = [], []
pooled_pred_event = {name: [] for name in APPROACH_NAMES}
pooled_pred_score = {name: [] for name in APPROACH_NAMES}

for csv_path in DATASET_CSV_PATHS:
    match_name = os.path.splitext(os.path.basename(csv_path))[0]
    print(f"\n{'=' * 60}\nEvaluating: {match_name}\n{'=' * 60}")

    df = load_dataset_csv(csv_path, loaded_model.label2id)
    texts = df["text"].tolist()
    true_event = df["true_event_id"].tolist()
    true_score = df["true_score"].tolist()

    predictions = run_all_approaches(texts, loaded_model.id2label, loaded_model, lexicon_model)

    classification_metrics, regression_metrics = {}, {}
    for approach in APPROACH_NAMES:
        pred_event = predictions[approach]["event_id"]
        pred_score = predictions[approach]["score"]

        classification_metrics[approach] = compute_classification_metrics(true_event, pred_event)
        regression_metrics[approach] = compute_regression_metrics(true_score, pred_score)

        pooled_pred_event[approach].extend(pred_event)
        pooled_pred_score[approach].extend(pred_score)

    pooled_true_event.extend(true_event)
    pooled_true_score.extend(true_score)

    print_metrics_table("Classification", classification_metrics, CLASSIFICATION_KEYS)
    print_metrics_table("Regression", regression_metrics, REGRESSION_KEYS)

    per_match_results[match_name] = {
        "classification": classification_metrics,
        "regression": regression_metrics,
    }

# --- Combined (pooled across all uploaded matches) ---
print(f"\n{'=' * 60}\nCOMBINED (all {len(DATASET_CSV_PATHS)} matches pooled)\n{'=' * 60}")
combined_classification, combined_regression = {}, {}
for approach in APPROACH_NAMES:
    combined_classification[approach] = compute_classification_metrics(
        pooled_true_event, pooled_pred_event[approach]
    )
    combined_regression[approach] = compute_regression_metrics(
        pooled_true_score, pooled_pred_score[approach]
    )

print_metrics_table("Classification", combined_classification, CLASSIFICATION_KEYS)
print_metrics_table("Regression", combined_regression, REGRESSION_KEYS)




Evaluating: dataset_match_1_Geo_vs_POR_WCA_2023

--- Classification ---
Approach                accuracy       precision          recall              f1 precision_macro    recall_macro        f1_macro
--------------------------------------------------------------------------------------------------------------------------------
ML Model Only             0.8096          0.8372          0.8096          0.8185          0.6341          0.6391          0.6205
Lexicon Only              0.2601          0.6512          0.2601          0.3430          0.1475          0.2276          0.1281
Hybrid Model              0.8046          0.8359          0.8046          0.8148          0.6274          0.6418          0.6165

--- Regression ---
Approach                     mse             mae
------------------------------------------------
ML Model Only             0.0609          0.2119
Lexicon Only              0.2632          0.4080
Hybrid Model              0.2636          0.4126

COMBINED (all 1 

In [13]:

# ============================================================
# CELL 13: Export results as JSON
#
# Schema mirrors commentary_analysis_system/data/output/evaluation_metrics.json
# ("Lexicon Only" / "ML Model Only" / "Hybrid Model" keys), extended
# with accuracy/precision_macro/recall_macro/f1_macro. Includes both
# the per-match breakdown and the combined/pooled result.
# ============================================================
from datetime import datetime, timezone

output = {
    "model_dir": os.path.basename(MODEL_ZIP_PATH),
    "datasets": DATASET_CSV_PATHS,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "hybrid_weights": {
        "highlight_lexicon_weight": HIGHLIGHT_LEXICON_WEIGHT,
        "event_lexicon_weight": EVENT_LEXICON_WEIGHT,
        "roberta_weight": ROBERTA_WEIGHT,
    },
    "per_match": per_match_results,
    "combined": {
        "classification": combined_classification,
        "regression": combined_regression,
    },
}

OUTPUT_PATH = "/content/evaluation_results.json"
with open(OUTPUT_PATH, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved results to {OUTPUT_PATH}")
files.download(OUTPUT_PATH)


Saved results to /content/evaluation_results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>